In [15]:
from pathlib import Path
import pandas as pd
import sys
import numpy as np

ROOT = Path('/Users/josh/Library/CloudStorage/GoogleDrive-jeverer@gmail.com/My Drive/London Sport/london_sport2')

df = pd.read_csv(ROOT / 'exploration' / 'data' / 'master_data' / '2016_to_2023_clustering_output_data.csv')
print(df.columns.tolist())
print(df.shape)

sys.path.append(str(ROOT / 'src' / 'stream_1' / 'forecasting'))
from forecasting_class import Forecast

['serial', 'year', 'month', 'calendar_year', 'calendar_month', 'wt_final', 'LCA_Class', 'LA_2023', 'active', 'Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10', 'WorkStat8', 'Child4', 'HHLiv9', 'Motiva_POP', 'motivd_POP', 'nadult', 'nchild', 'health', 'comm1', 'anxious', 'happy', 'lifesat', 'lone', 'DVBMI', 'FruitVegPor', 'READYAB1_POP', 'CULFRQ_1_9_POP', 'VolAny', 'VolCnt', 'VolDur', 'VolFrqB_Pop', 'volint1', 'volint2', 'volint3', 'volint4', 'volint5', 'volint6', 'volint7', 'MEMS7_ALL', 'MEMS7_SPORTCOUNT_A01', 'MEMS7_IN_SPORTCOUNT_A01', 'MEMS7_OUT_SPORTCOUNT_A01', 'MEMS7_FITNESS_B06', 'MEMS7_WALKALL_C01', 'MEMS7_CYCALL_C02', 'MEMS7_ACTTRAV_C03', 'MEMS7_DANCEALL_C04', 'MEMS7_TEAMSPORT_C05', 'MEMS7_RACKETSPORT_C06', 'MEMS7_ADVWATERSPORT_C07', 'MEMS7_LEISURE_C08', 'MEMS7_COMBATTARGET_C09', 'MEMS7_WINTER_C10', 'MEMS7_RUNATHMULTI_C11', 'ACT7GR_ALL', 'ACT7GR_SPORTCOUNT_A01', 'Number_Activities', 'CLUB_SPORTCOUNT_A01', 'Number_Club']
(85659, 65)


In [16]:
df = pd.read_csv(ROOT / 'exploration' / 'data' / 'master_data' / '2016_to_2023_clustering_output_data.csv')

# quarterly
df_lca_quarterly = df[df['LCA_Class'].notna()].copy()
df_lca_quarterly['year'] = df_lca_quarterly['year'].str.split('/').str[1].astype(int) + 2000
df_lca_quarterly['month'] = ((df_lca_quarterly['month'].astype(int) - 3) % 12) + 1
df_lca_quarterly['quarter'] = pd.cut(df_lca_quarterly['month'], bins=[0,3,6,9,12], labels=[1,2,3,4])
df_lca_quarterly = df_lca_quarterly.groupby(['LCA_Class', 'year', 'quarter']).agg(MEMS7_ALL=('MEMS7_ALL', 'mean')).reset_index()

# monthly
df_lca_monthly = df[df['LCA_Class'].notna()].copy()
df_lca_monthly['year'] = df_lca_monthly['year'].str.split('/').str[1].astype(int) + 2000
df_lca_monthly['month'] = ((df_lca_monthly['month'].astype(int) - 3) % 12) + 1
df_lca_monthly.loc[df_lca_monthly['month'].isin([11,12]), 'year'] = df_lca_monthly['year'] - 1
df_lca_monthly = df_lca_monthly.groupby(['LCA_Class', 'year', 'month']).agg(MEMS7_ALL=('MEMS7_ALL', 'mean')).reset_index()

In [17]:
from forecasting_class import Forecast
print("quarter")
sqc = Forecast(df_lca_quarterly, forecast_steps=20, group_col='LCA_Class')
sqc.sarima()
print(sqc.get_mae('sarima'))

print("month")
smc = Forecast(df_lca_monthly, forecast_steps=60, group_col='LCA_Class')
smc.sarima()
print(smc.get_mae('sarima'))

quarter
nan
month
nan


In [18]:
# print("---------------------------")
# print("----- YEARLY FORECAST -----")
# print("---------------------------")
# print(np.mean(results_lca['mape']))
# xtick_positions = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# xtick_labels = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
# plot_cluster_forecasts(results_lca, t_train_cutoff=5, xtick_positions=xtick_positions, xtick_labels=xtick_labels, group_col='LCA_Class', UNCERTAINTY=True)

In [19]:
# print("------------------------------")
# print("----- QUARTERLY FORECAST -----")
# print("------------------------------")
# print(np.mean(results_lca_q['mape']))
# xtick_positions = [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
# xtick_labels = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
# plot_cluster_forecasts(results_lca_q, t_train_cutoff=20, xtick_positions=xtick_positions, xtick_labels=xtick_labels, group_col='LCA_Class', UNCERTAINTY=True)

In [20]:
# print("-------------------------------------")
# print("----- QUARTERLY SARIMA FORECAST -----")
# print("-------------------------------------")
# print(np.mean(results_sarima['mape']))
# xtick_positions = [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40]
# xtick_labels = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
# plot_cluster_forecasts(results_sarima, t_train_cutoff=20, xtick_positions=xtick_positions, xtick_labels=xtick_labels, group_col='LCA_Class')

In [21]:
df_lca_annual = df[df['LCA_Class'].notna()].copy()
df_lca_annual['year'] = df_lca_annual['year'].str.split('/').str[1].astype(int) + 2000
df_lca_annual = df_lca_annual.groupby(['LCA_Class', 'year']).agg(MEMS7_ALL=('MEMS7_ALL', 'mean')).reset_index()


In [22]:
lca_year = Forecast(df_lca_annual, forecast_steps=4, group_col='LCA_Class')
lca_year.bayesian_ridge()
print(lca_year.get_mae('bayesian_ridge'))

lca_quarter = Forecast(df_lca_quarterly, forecast_steps=16, group_col='LCA_Class')
lca_quarter.sarima()
print(lca_quarter.get_mae('sarima'))

nan
nan


In [23]:
rows = []
for _, row in lca_year.bayesian_ridge_forecast.iterrows():
    y_train = list(row['y_train'])
    y_forecast = list(row['y_forecast'])
    for t, actual in enumerate(y_train):
        rows.append({'LCA_Class': row['LCA_Class'], 't': t, 'MEMS_actual': actual, 'MEMS_bayesian': None, 'period': 'train'})
    rows.append({'LCA_Class': row['LCA_Class'], 't': len(y_train), 'MEMS_actual': y_train[-1], 'MEMS_bayesian': y_forecast[0], 'period': 'forecast'})
    for t, forecast in enumerate(y_forecast[1:], start=len(y_train)+1):
        rows.append({'LCA_Class': row['LCA_Class'], 't': t, 'MEMS_bayesian': forecast, 'MEMS_actual': None, 'period': 'forecast'})

class_labels = {
    0: 'Middle-aged Working Fathers', 1: 'Highly Educated Working Mothers',
    2: 'Professional Fathers', 3: 'Part-time Working Professional Mothers',
    4: 'Later-career Middle Class Workers', 5: 'Highly Educated Early Retirees',
    6: 'Professional Women from Ethnic Minority Backgrounds', 7: 'British-born Retirees',
    8: 'Students in Shared Housing', 9: 'Young Middle Class Workers',
    10: 'Motivated Young Professional Men', 11: 'Graduate Professionals in Shared Housing',
    12: 'Later-career Professional Women', 13: 'Unemployed Adults from Deprived Backgrounds',
    14: 'Long-term Sick and Disabled Adults', 15: 'Mid-career Professionals Living Alone',
    16: 'Mothers and Carers from Deprived Backgrounds', 17: 'Young Professional Women',
    18: 'Older Parents Approaching Retirement', 19: 'Independent Older Retirees',
    20: 'International Educated Professionals', 21: 'Young Students Living with Parents',
    22: 'Older Retirees with Fewer Qualifications', 23: 'Later-career Professional Men',
    24: 'Disabled Older Retirees', 25: 'Adults from Deprived Backgrounds Living with Parents',
    26: 'Young Professionals Living with Parents'
}
df_year = pd.DataFrame(rows)
df_year['LCA_Label'] = df_year['LCA_Class'].map(class_labels)
df_year.to_csv('tableau_cluster_yearly.csv', index=False)

In [24]:
rows = []
for _, row in lca_quarter.sarima_forecast.iterrows():
    y_train = list(row['y_train'])
    y_forecast = list(row['y_forecast'])
    for t, actual in enumerate(y_train):
        rows.append({'LCA_Class': row['LCA_Class'], 't': t, 'MEMS_actual': actual, 'MEMS_sarima': None, 'period': 'train'})
    rows.append({'LCA_Class': row['LCA_Class'], 't': len(y_train), 'MEMS_actual': y_train[-1], 'MEMS_sarima': y_forecast[0], 'period': 'forecast'})
    for t, forecast in enumerate(y_forecast[1:], start=len(y_train)+1):
        rows.append({'LCA_Class': row['LCA_Class'], 't': t, 'MEMS_sarima': forecast, 'MEMS_actual': None, 'period': 'forecast'})

df_quarter = pd.DataFrame(rows)
df_quarter['LCA_Label'] = df_quarter['LCA_Class'].map(class_labels)
df_quarter.to_csv('tableau_cluster_quarterly.csv', index=False)

In [25]:
import sys
sys.path.append(str(ROOT))
from src.loading_data.data_catalogue import DataCatalogue
dc = DataCatalogue()

df_valid = df[df['LCA_Class'].notna()].copy()
df_valid['% Active'] = df_valid['active'] * 100
df_valid['% Volunteering'] = df_valid['VolAny'] * 100
df_valid['% Ethnic Minority'] = (
    (df_valid['Eth7'].isin([3, 4, 5, 6, 7])) & df_valid['Eth7'].notna()
).astype(float) * 100
df_valid['% Disability'] = df_valid['Disab2_POP'] * 100
df_valid['Deprivation'] = ((10 - df_valid['IMD10'].replace(0, np.nan)) / 9) * 100
df_valid['Community Trust'] = ((df_valid['comm1'] - 1) / 4) * 100
df_valid['Enjoyment'] = ((5 - df_valid['Motiva_POP'].replace(0, np.nan) - 1) / 3) * 100

df_valid['LA_Name'] = df_valid['LA_2023'].map(dc.get_data_dict()['LA_2023']['value_labels'])
df_valid['LA_Name'] = df_valid['LA_Name'].str.replace(' upon Thames', '')
df_valid = df_valid[df_valid['LA_Name'] != 'City of London']

class_labels = {
    0: 'Middle-aged Working Fathers',
    1: 'Highly Educated Working Mothers',
    2: 'Professional Fathers',
    3: 'Part-time Working Professional Mothers',
    4: 'Later-career Middle Class Workers',
    5: 'Highly Educated Early Retirees',
    6: 'Professional Women from Ethnic Minority Backgrounds',
    7: 'British-born Retirees',
    8: 'Students in Shared Housing',
    9: 'Young Middle Class Workers',
    10: 'Motivated Young Professional Men',
    11: 'Graduate Professionals in Shared Housing',
    12: 'Later-career Professional Women',
    13: 'Unemployed Adults from Deprived Backgrounds',
    14: 'Long-term Sick and Disabled Adults',
    15: 'Mid-career Professionals Living Alone',
    16: 'Mothers and Carers from Deprived Backgrounds',
    17: 'Young Professional Women',
    18: 'Older Parents Approaching Retirement',
    19: 'Independent Older Retirees',
    20: 'International Educated Professionals',
    21: 'Young Students Living with Parents',
    22: 'Older Retirees with Fewer Qualifications',
    23: 'Later-career Professional Men',
    24: 'Disabled Older Retirees',
    25: 'Adults from Deprived Backgrounds Living with Parents',
    26: 'Young Professionals Living with Parents'
}



borough_cols = ['% Active', 'Deprivation', '% Volunteering', 'Community Trust', '% Ethnic Minority']
london_means_b = df_valid[borough_cols].mean()
borough_means = df_valid.groupby('LA_Name')[borough_cols].mean()

rows = []
for name, row in borough_means.iterrows():
    for var, val in row.items():
        rows.append({'LA Name': name, 'Variable': var, 'Value': round(val, 2), 'London_Mean': round(london_means_b[var], 2)})
pd.DataFrame(rows).to_csv('tableau_borough_demographics_dotplot.csv', index=False)

cluster_cols = ['% Active', 'Enjoyment', 'Community Trust', '% Disability', 'Deprivation']
london_means_c = df_valid[cluster_cols].mean()
cluster_means = df_valid.groupby('LCA_Class')[cluster_cols].mean()

rows = []
for cls, row in cluster_means.iterrows():
    for var, val in row.items():
        rows.append({'LCA_Class': int(cls), 'LCA_Label': class_labels[int(cls)], 'Variable': var, 'Value': round(val, 2), 'London_Mean': round(london_means_c[var], 2)})
pd.DataFrame(rows).to_csv('tableau_cluster_demographics_dotplot.csv', index=False)

In [26]:
print(df_valid['Motiva_POP'].value_counts().sort_index())

Motiva_POP
0    29264
1    34227
2    13047
3     5775
4     1857
Name: count, dtype: int64


In [27]:
print([n for n in borough_means.index if 'rich' in n.lower() or 'king' in n.lower()])

['Barking and Dagenham', 'Kingston', 'Richmond']
